In [1]:
!pip install fastapi uvicorn nest_asyncio pyngrok transformers torch


In [2]:
!pip install python-multipart

In [3]:
!ngrok config add-authtoken 2w2nqCVkpjmGtOSPsGCh2hYBYnP_3kpMz3rhgFEsk6fPsH9DT

Authtoken saved to configuration file: C:\Users\lenovo\AppData\Local/ngrok/ngrok.yml


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path to your saved model
model_path = "./gemma_large"  # Or "/kaggle/working/gemma-finetuned-final" if still on Kaggle

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")

# Set model to evaluation mode
model.eval()

# Input prompt
prompt = "Once upon a time"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,  # Adjust as needed
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        do_sample=True
    )

# Decode and print the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


c:\Users\lenovo\anaconda3\envs\nik\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Once upon a time, in a small village nestled amidst rolling hills, lived a young girl named Anya. She was known for her kindness, her gentle spirit, and her ability to make the most of any situation. One day, a traveling merchant arrived in the village,


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load your fine-tuned model
model_path = "./gemma_large"  # Update path if needed
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()


def generate_text(prompt_text):
    try:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.15
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return {"answer": response}
    except Exception as e:
        return {"error": str(e)}


# FastAPI server setup
from fastapi import FastAPI, Form
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI()

@app.post("/generate/")
async def generate_endpoint(prompt: str = Form(...)):
    try:
        result = generate_text(prompt)
        return JSONResponse(content=result)
    except Exception as e:
        return JSONResponse(content={"error": str(e)}, status_code=500)

@app.get("/")
async def root():
    return {"message": "Gemma Fine-tuned Model API is running. Send a POST request to /generate/ with a 'prompt' in the form data."}


# Setup Ngrok tunnel and run the server
if __name__ == "__main__":
    ngrok_tunnel = ngrok.connect(8000)
    print("Public URL:", ngrok_tunnel.public_url)
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)


Public URL: https://8fa3-14-139-239-178.ngrok-free.app


INFO:     Started server process [18988]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     14.139.239.178:0 - "GET / HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


INFO:     14.139.239.178:0 - "POST /generate/ HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [18988]
